# Notebook 02 — Data Preprocessing Pipeline

Covers: null handling, duplicate removal, encoding, scaling, 80/20 stratified split, and leakage prevention.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.malaria_forecast.config import load_config
from src.malaria_forecast.data_loader import load_raw_dataset
from src.malaria_forecast.preprocessing import preprocess_data
from src.malaria_forecast.features import ALL_NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COLUMN

config = load_config('../config/config.yaml')
df = load_raw_dataset('../dataset/Malaria_Dataset.csv')
print('Raw shape:', df.shape)

## 1. Missing Values Check

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal nulls: {df.isnull().sum().sum()}')

## 2. Duplicate Removal

In [ ]:
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f'Shape after dedup: {df_clean.shape}')

## 3. Class Imbalance

In [ ]:
counts = df_clean[TARGET_COLUMN].value_counts().sort_index()
total = len(df_clean)
for cls, cnt in counts.items():
    print(f'  Class {cls}: {cnt} ({100*cnt/total:.1f}%)')
print(f'\nImbalance ratio: {counts[1]/counts[0]:.2f}:1 (positive:negative)')
print('→ class_weight="balanced" applied to all models')

## 4. Run Full Preprocessing Pipeline

In [ ]:
result = preprocess_data(df_clean, config)

X_train = result['X_train']
X_test  = result['X_test']
y_train = result['y_train']
y_test  = result['y_test']

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('\nFeature order:', result['final_feature_order'])
print('\nImputation defaults:', result['imputation_defaults'])

## 5. Processed Data Sample

In [ ]:
print('X_train (first 5 rows):')
X_train.head()


## 6. Save Processed Dataset

In [ ]:
import pandas as pd
from pathlib import Path

out_path = Path('../data/processed/malaria_processed.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
full = X_train.copy()
full[TARGET_COLUMN] = y_train.values
full.to_csv(out_path, index=False)
print(f'Saved processed training set to: {out_path}')